# Cross-View Normal Consistency with COLMAP
This notebook evaluates cross-view consistency of estimated normal maps by lifting per-view normals to world coordinates using COLMAP poses, resolving sign ambiguity, and measuring per-track angular residuals.

In [ ]:
from pathlib import Path
import os

DATASET_ROOT = Path("/home/roman/ba/datasets/ref_real/sedan")
SPARSE_ROOT = DATASET_ROOT / "sparse/0"
NORMALS_ROOT = DATASET_ROOT / "normals"

# Normal decoding conventions
CHANNEL_ORDER = "RGB"  # "RGB" or "BGR"
FLIP_Y = False  # Set True if your normal convention uses opposite Y

# Evaluation controls
MAX_POINTS = None  # None = use all COLMAP 3D points
MIN_OBS_PER_POINT = 2  # Points with fewer obs get NaN consistency stats
PRINT_EVERY_IMAGES = 25

# Optional analysis / visualization
RUN_PAIRWISE_ANALYSIS = False
PAIRWISE_MIN_SHARED = 80
ENABLE_OPEN3D_VIS = True  # Master switch for Open3D visualization
OPEN3D_VIS_BACKEND = "auto"  # "auto", "notebook", or "window"
OPEN3D_MAX_VECTOR_COUNT = 800

assert SPARSE_ROOT.exists(), f"Missing COLMAP model dir: {SPARSE_ROOT}"
assert NORMALS_ROOT.exists(), f"Missing normals dir: {NORMALS_ROOT}"

print("DATASET_ROOT:", DATASET_ROOT)
print("SPARSE_ROOT :", SPARSE_ROOT)
print("NORMALS_ROOT:", NORMALS_ROOT)
print("ENABLE_OPEN3D_VIS:", ENABLE_OPEN3D_VIS)
print("OPEN3D_VIS_BACKEND:", OPEN3D_VIS_BACKEND)


In [ ]:
import math
from itertools import combinations
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from easyvolcap.easyvolcap.utils.colmap_utils import read_model, qvec2rotmat


def is_notebook_environment() -> bool:
    try:
        from IPython import get_ipython

        shell = get_ipython()
        if shell is None:
            return False
        return shell.__class__.__name__ == "ZMQInteractiveShell"
    except Exception:
        return False


IN_NOTEBOOK = is_notebook_environment()

try:
    import open3d as o3d

    OPEN3D_AVAILABLE = True
except Exception as exc:
    OPEN3D_AVAILABLE = False
    o3d = None
    print(f"Open3D unavailable: {exc}")

OPEN3D_WEB_DRAW_AVAILABLE = False
o3d_web_draw = None
if OPEN3D_AVAILABLE:
    try:
        from open3d.web_visualizer import draw as o3d_web_draw

        OPEN3D_WEB_DRAW_AVAILABLE = True
    except Exception as exc:
        print(f"Open3D notebook draw unavailable: {exc}")

print("IN_NOTEBOOK:", IN_NOTEBOOK)
if OPEN3D_AVAILABLE:
    print("Open3D version:", getattr(o3d, "__version__", "unknown"))
print("OPEN3D_WEB_DRAW_AVAILABLE:", OPEN3D_WEB_DRAW_AVAILABLE)

np.set_printoptions(precision=4, suppress=True)


In [ ]:
cameras, images, points3D = read_model(str(SPARSE_ROOT))
images_by_id = dict(sorted(images.items(), key=lambda kv: kv[0]))
points3D_by_id = points3D

print(f"Cameras : {len(cameras)}")
print(f"Images  : {len(images_by_id)}")
print(f"Points3D: {len(points3D_by_id)}")

first_cam = next(iter(cameras.values()))
print("Camera model:", first_cam.model, "resolution:", (first_cam.width, first_cam.height))

print("First 5 COLMAP image entries:")
for image_id, image in list(images_by_id.items())[:5]:
    num_triangulated = int((image.point3D_ids >= 0).sum())
    print(f"  {image_id:4d} | {image.name:<20} | triangulated obs: {num_triangulated}")

In [ ]:
def safe_normalize(vec: np.ndarray, eps: float = 1e-8) -> Optional[np.ndarray]:
    vec = np.asarray(vec, dtype=np.float64)
    n = np.linalg.norm(vec)
    if not np.isfinite(n) or n < eps:
        return None
    return vec / n


def bilinear_sample_rgb(image_rgb: np.ndarray, u: float, v: float) -> Optional[np.ndarray]:
    h, w = image_rgb.shape[:2]
    if not (0.0 <= u <= (w - 1) and 0.0 <= v <= (h - 1)):
        return None

    x0 = int(np.floor(u))
    y0 = int(np.floor(v))
    x1 = min(x0 + 1, w - 1)
    y1 = min(y0 + 1, h - 1)

    dx = u - x0
    dy = v - y0

    top = (1.0 - dx) * image_rgb[y0, x0] + dx * image_rgb[y0, x1]
    bot = (1.0 - dx) * image_rgb[y1, x0] + dx * image_rgb[y1, x1]
    return (1.0 - dy) * top + dy * bot


def decode_normal_rgb(
    rgb: np.ndarray,
    channel_order: str = "RGB",
    flip_y: bool = False,
) -> Optional[np.ndarray]:
    if rgb is None or not np.all(np.isfinite(rgb)):
        return None

    rgb = np.asarray(rgb, dtype=np.float64)
    if channel_order.upper() == "BGR":
        rgb = rgb[::-1]
    elif channel_order.upper() != "RGB":
        raise ValueError(f"Unsupported channel order: {channel_order}")

    n_cam = 2.0 * (rgb / 255.0) - 1.0
    if flip_y:
        n_cam[1] *= -1.0
    return safe_normalize(n_cam)


def angle_deg(a: np.ndarray, b: np.ndarray) -> float:
    dot = float(np.clip(np.dot(a, b), -1.0, 1.0))
    return float(np.degrees(np.arccos(dot)))


def build_pose_cache(images_dict: Dict[int, object]) -> Dict[int, Dict[str, np.ndarray]]:
    pose_cache = {}
    for image_id, image in images_dict.items():
        R = qvec2rotmat(image.qvec)
        t = np.asarray(image.tvec, dtype=np.float64)
        C = -R.T @ t
        pose_cache[image_id] = {"R": R, "t": t, "C": C}
    return pose_cache

In [ ]:
VALID_NORMAL_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".exr")


@dataclass
class NormalIndex:
    files: List[Path]
    by_rel: Dict[str, List[Path]]
    by_name: Dict[str, List[Path]]
    by_stem: Dict[str, List[Path]]
    id_fallback: Dict[int, Path]


def list_normal_files(normals_root: Path) -> List[Path]:
    files = []
    for p in normals_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_NORMAL_EXTS:
            files.append(p)
    return sorted(files)


def _group_paths(paths: List[Path], key_fn) -> Dict[str, List[Path]]:
    out = defaultdict(list)
    for p in paths:
        out[key_fn(p)].append(p)
    return dict(out)


def _choose_unique(candidates: List[Path]) -> Optional[Path]:
    if len(candidates) == 1:
        return candidates[0]
    return None


def _choose_from_dir(dir_files: List[Path]) -> Optional[Path]:
    if not dir_files:
        return None
    zero = [p for p in dir_files if p.stem == "000000"]
    if len(zero) == 1:
        return zero[0]
    if len(zero) > 1:
        return sorted(zero)[0]
    if len(dir_files) == 1:
        return dir_files[0]
    return sorted(dir_files)[0]


def build_normal_index(normals_root: Path) -> NormalIndex:
    files = list_normal_files(normals_root)
    by_rel = _group_paths(files, lambda p: p.relative_to(normals_root).as_posix().lower())
    by_name = _group_paths(files, lambda p: p.name.lower())
    by_stem = _group_paths(files, lambda p: p.stem.lower())

    id_fallback = {}
    if normals_root.exists():
        for child in sorted(normals_root.iterdir()):
            if not child.is_dir() or not child.name.isdigit():
                continue
            image_id = int(child.name) + 1
            child_files = [
                p
                for p in child.rglob("*")
                if p.is_file() and p.suffix.lower() in VALID_NORMAL_EXTS
            ]
            chosen = _choose_from_dir(sorted(child_files))
            if chosen is not None and image_id not in id_fallback:
                id_fallback[image_id] = chosen

    return NormalIndex(files=files, by_rel=by_rel, by_name=by_name, by_stem=by_stem, id_fallback=id_fallback)


def match_normal_map(image_name: str, image_id: int, normal_index: NormalIndex) -> Tuple[Optional[Path], str]:
    rel = Path(image_name).as_posix()
    rel_key = rel.lower()
    base = Path(image_name).name
    base_key = base.lower()
    stem_key = Path(image_name).stem.lower()

    def try_rel(key: str, tag: str) -> Tuple[Optional[Path], str]:
        if key in normal_index.by_rel:
            p = _choose_unique(normal_index.by_rel[key])
            if p is not None:
                return p, tag
            return None, f"ambiguous:{tag}"
        return None, ""

    p, tag = try_rel(rel_key, "relative")
    if p is not None:
        return p, tag

    rel_path = Path(rel)
    for ext in VALID_NORMAL_EXTS:
        alt_rel = rel_path.with_suffix(ext).as_posix().lower()
        p, tag = try_rel(alt_rel, "relative_ext_swap")
        if p is not None:
            return p, tag

    if base_key in normal_index.by_name:
        p = _choose_unique(normal_index.by_name[base_key])
        if p is not None:
            return p, "basename"

    base_path = Path(base)
    for ext in VALID_NORMAL_EXTS:
        alt_base = base_path.with_suffix(ext).name.lower()
        if alt_base in normal_index.by_name:
            p = _choose_unique(normal_index.by_name[alt_base])
            if p is not None:
                return p, "basename_ext_swap"

    if stem_key in normal_index.by_stem:
        p = _choose_unique(normal_index.by_stem[stem_key])
        if p is not None:
            return p, "stem"

    if image_id in normal_index.id_fallback:
        return normal_index.id_fallback[image_id], "image_id_minus_one_subfolder"

    return None, "unmatched"

In [ ]:
normal_index = build_normal_index(NORMALS_ROOT)
print(f"Indexed {len(normal_index.files)} normal maps")
print(f"Subfolder image_id fallback entries: {len(normal_index.id_fallback)}")

match_rows = []
normal_path_by_image_id = {}
for image_id, image in images_by_id.items():
    normal_path, match_method = match_normal_map(image.name, image_id, normal_index)
    if normal_path is not None:
        normal_path_by_image_id[image_id] = normal_path
    match_rows.append(
        {
            "image_id": image_id,
            "image_name": image.name,
            "normal_path": str(normal_path) if normal_path is not None else None,
            "match_method": match_method,
            "matched": normal_path is not None,
        }
    )

match_df = pd.DataFrame(match_rows)
print("Matched images:", int(match_df["matched"].sum()), "/", len(match_df))
display(match_df["match_method"].value_counts().rename_axis("match_method").to_frame("count"))

unmatched_df = match_df[~match_df["matched"]].copy()
if len(unmatched_df):
    print("Unmatched frames:", len(unmatched_df))
    display(unmatched_df.head(20))
else:
    print("All COLMAP images matched to a normal map.")

In [ ]:
def evaluate_normal_consistency(
    images_dict: Dict[int, object],
    points3D_dict: Dict[int, object],
    normal_paths: Dict[int, Path],
    channel_order: str = "RGB",
    flip_y: bool = False,
    max_points: Optional[int] = None,
    min_obs_per_point: int = 2,
    print_every_images: int = 25,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    pose_cache = build_pose_cache(images_dict)

    if max_points is None:
        allowed_points = None
    else:
        kept = sorted(points3D_dict.keys())[: int(max_points)]
        allowed_points = set(kept)
        print(f"Using first {len(allowed_points)} points3D due to MAX_POINTS")

    obs_rows: List[dict] = []
    track_normals: Dict[int, List[np.ndarray]] = defaultdict(list)
    track_obs_indices: Dict[int, List[int]] = defaultdict(list)

    total_images = len(images_dict)
    for idx, (image_id, image) in enumerate(images_dict.items(), start=1):
        if print_every_images and (idx == 1 or idx % print_every_images == 0 or idx == total_images):
            print(f"[{idx:4d}/{total_images}] image_id={image_id} name={image.name}")

        normal_path = normal_paths.get(image_id)
        if normal_path is None:
            continue

        with Image.open(normal_path) as im:
            normal_rgb = np.asarray(im.convert("RGB"), dtype=np.float32)

        pose = pose_cache[image_id]
        R = pose["R"]
        C = pose["C"]

        xys = np.asarray(image.xys, dtype=np.float64)
        point_ids = np.asarray(image.point3D_ids, dtype=np.int64)

        for k in range(point_ids.shape[0]):
            point_id = int(point_ids[k])
            if point_id < 0:
                continue
            if allowed_points is not None and point_id not in allowed_points:
                continue
            point = points3D_dict.get(point_id)
            if point is None:
                continue

            u = float(xys[k, 0])
            v = float(xys[k, 1])
            rgb = bilinear_sample_rgb(normal_rgb, u, v)
            if rgb is None:
                continue

            n_cam = decode_normal_rgb(rgb, channel_order=channel_order, flip_y=flip_y)
            if n_cam is None:
                continue

            n_world = safe_normalize(R.T @ n_cam)
            if n_world is None:
                continue

            P = np.asarray(point.xyz, dtype=np.float64)
            view_dir = safe_normalize(C - P)
            if view_dir is None:
                continue

            flip_flag = False
            if float(np.dot(n_world, view_dir)) < 0.0:
                n_world = -n_world
                flip_flag = True

            obs_index = len(obs_rows)
            obs_rows.append(
                {
                    "point3D_id": point_id,
                    "image_id": image_id,
                    "image_name": image.name,
                    "u": u,
                    "v": v,
                    "n_cam_x": float(n_cam[0]),
                    "n_cam_y": float(n_cam[1]),
                    "n_cam_z": float(n_cam[2]),
                    "n_world_x": float(n_world[0]),
                    "n_world_y": float(n_world[1]),
                    "n_world_z": float(n_world[2]),
                    "flip_by_view_dir": bool(flip_flag),
                    "angle_to_consensus_deg": np.nan,
                    "consensus_nx": np.nan,
                    "consensus_ny": np.nan,
                    "consensus_nz": np.nan,
                }
            )
            track_normals[point_id].append(n_world)
            track_obs_indices[point_id].append(obs_index)

    point_rows: List[dict] = []
    for point_id, normals in track_normals.items():
        n = np.asarray(normals, dtype=np.float64)
        num_obs = int(n.shape[0])
        if num_obs == 0:
            continue

        consensus = safe_normalize(n.sum(axis=0))
        if consensus is None:
            consensus = safe_normalize(n[0])
        if consensus is None:
            continue

        dots = np.clip(n @ consensus, -1.0, 1.0)
        angles = np.degrees(np.arccos(dots))
        dispersion = float(1.0 - np.linalg.norm(n.sum(axis=0)) / max(num_obs, 1))

        if num_obs < int(min_obs_per_point):
            angles_for_eval = np.full_like(angles, np.nan, dtype=np.float64)
            mean_angle = np.nan
            median_angle = np.nan
        else:
            angles_for_eval = angles
            mean_angle = float(np.mean(angles))
            median_angle = float(np.median(angles))

        for obs_idx, ang in zip(track_obs_indices[point_id], angles_for_eval):
            obs_rows[obs_idx]["angle_to_consensus_deg"] = float(ang) if np.isfinite(ang) else np.nan
            obs_rows[obs_idx]["consensus_nx"] = float(consensus[0])
            obs_rows[obs_idx]["consensus_ny"] = float(consensus[1])
            obs_rows[obs_idx]["consensus_nz"] = float(consensus[2])

        point_rows.append(
            {
                "point3D_id": point_id,
                "num_valid_observations": num_obs,
                "mean_angle_deg": mean_angle,
                "median_angle_deg": median_angle,
                "dispersion_proxy": dispersion,
                "consensus_nx": float(consensus[0]),
                "consensus_ny": float(consensus[1]),
                "consensus_nz": float(consensus[2]),
            }
        )

    obs_df = pd.DataFrame(obs_rows)
    point_df = pd.DataFrame(point_rows)

    if len(obs_df):
        image_df = (
            obs_df.groupby(["image_id", "image_name"], as_index=False)
            .agg(
                num_evaluated_observations=("point3D_id", "size"),
                num_with_consensus=("angle_to_consensus_deg", lambda x: int(np.isfinite(x).sum())),
                mean_angle_deg=("angle_to_consensus_deg", "mean"),
                median_angle_deg=("angle_to_consensus_deg", "median"),
            )
            .sort_values("image_id")
            .reset_index(drop=True)
        )
    else:
        image_df = pd.DataFrame(
            columns=[
                "image_id",
                "image_name",
                "num_evaluated_observations",
                "num_with_consensus",
                "mean_angle_deg",
                "median_angle_deg",
            ]
        )

    return obs_df, point_df, image_df

In [ ]:
obs_df, point_df, image_df = evaluate_normal_consistency(
    images_dict=images_by_id,
    points3D_dict=points3D_by_id,
    normal_paths=normal_path_by_image_id,
    channel_order=CHANNEL_ORDER,
    flip_y=FLIP_Y,
    max_points=MAX_POINTS,
    min_obs_per_point=MIN_OBS_PER_POINT,
    print_every_images=PRINT_EVERY_IMAGES,
)

print("Observation-level rows:", len(obs_df))
print("Point-level rows      :", len(point_df))
print("Image-level rows      :", len(image_df))

display(obs_df.head(5))
display(point_df.head(5))
display(image_df.head(5))

In [ ]:
def summarize_angles(angle_series: pd.Series) -> pd.Series:
    values = angle_series.to_numpy(dtype=np.float64)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return pd.Series(dtype=np.float64)

    stats = {
        "count": float(values.size),
        "mean_deg": float(np.mean(values)),
        "median_deg": float(np.median(values)),
        "p50_deg": float(np.percentile(values, 50)),
        "p75_deg": float(np.percentile(values, 75)),
        "p90_deg": float(np.percentile(values, 90)),
        "p95_deg": float(np.percentile(values, 95)),
    }
    return pd.Series(stats)


global_stats = summarize_angles(obs_df["angle_to_consensus_deg"])
print("Global angular-error stats (degrees):")
display(global_stats.to_frame("value"))

valid_image_stats = image_df[np.isfinite(image_df["median_angle_deg"])].copy()
worst_images = valid_image_stats.sort_values("median_angle_deg", ascending=False).head(10)
best_images = valid_image_stats.sort_values("median_angle_deg", ascending=True).head(10)

print("Worst images by median angular error:")
display(worst_images[["image_id", "image_name", "num_with_consensus", "median_angle_deg", "mean_angle_deg"]])

print("Best images by median angular error:")
display(best_images[["image_id", "image_name", "num_with_consensus", "median_angle_deg", "mean_angle_deg"]])

In [ ]:
valid_angles = obs_df["angle_to_consensus_deg"].to_numpy(dtype=np.float64)
valid_angles = valid_angles[np.isfinite(valid_angles)]

if valid_angles.size == 0:
    print("No valid angular errors to plot yet.")
else:
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))

    ax[0].hist(valid_angles, bins=80, color="#1f77b4", alpha=0.9)
    ax[0].set_title("Histogram of Observation Angular Errors")
    ax[0].set_xlabel("Angle to consensus (deg)")
    ax[0].set_ylabel("Count")
    ax[0].grid(True, alpha=0.25)

    show_worst = valid_image_stats.sort_values("median_angle_deg", ascending=False).head(12)
    show_worst = show_worst.sort_values("median_angle_deg", ascending=True)
    ax[1].barh(show_worst["image_name"], show_worst["median_angle_deg"], color="#d62728")
    ax[1].set_title("Worst Images by Median Error")
    ax[1].set_xlabel("Median angle (deg)")
    ax[1].grid(True, axis="x", alpha=0.25)

    plt.tight_layout()
    plt.show()

In [ ]:
def compute_pairwise_frame_consistency(obs_df: pd.DataFrame, min_shared: int = 80) -> pd.DataFrame:
    needed = ["point3D_id", "image_id", "image_name", "n_world_x", "n_world_y", "n_world_z"]
    for c in needed:
        if c not in obs_df.columns:
            raise KeyError(f"Missing column: {c}")

    per_pair_sum = defaultdict(float)
    per_pair_count = defaultdict(int)
    image_names = {}

    grouped = obs_df.groupby("point3D_id", sort=False)
    for _, g in grouped:
        if len(g) < 2:
            continue

        entries = []
        for row in g.itertuples(index=False):
            n = np.array([row.n_world_x, row.n_world_y, row.n_world_z], dtype=np.float64)
            n = safe_normalize(n)
            if n is None:
                continue
            entries.append((int(row.image_id), str(row.image_name), n))

        if len(entries) < 2:
            continue

        for (i_id, i_name, n_i), (j_id, j_name, n_j) in combinations(entries, 2):
            if i_id == j_id:
                continue
            a, b = (i_id, j_id) if i_id < j_id else (j_id, i_id)
            n_a, n_b = (n_i, n_j) if i_id < j_id else (n_j, n_i)
            key = (a, b)
            per_pair_sum[key] += angle_deg(n_a, n_b)
            per_pair_count[key] += 1
            image_names[a] = i_name if i_id < j_id else j_name
            image_names[b] = j_name if i_id < j_id else i_name

    rows = []
    for (a, b), cnt in per_pair_count.items():
        if cnt < int(min_shared):
            continue
        rows.append(
            {
                "image_id_a": a,
                "image_name_a": image_names.get(a, str(a)),
                "image_id_b": b,
                "image_name_b": image_names.get(b, str(b)),
                "shared_points": cnt,
                "mean_pair_angle_deg": per_pair_sum[(a, b)] / cnt,
            }
        )

    pair_df = pd.DataFrame(rows)
    if len(pair_df):
        pair_df = pair_df.sort_values(
            ["mean_pair_angle_deg", "shared_points"],
            ascending=[False, False],
        ).reset_index(drop=True)
    return pair_df


if RUN_PAIRWISE_ANALYSIS:
    pairwise_df = compute_pairwise_frame_consistency(obs_df, min_shared=PAIRWISE_MIN_SHARED)
    print("Pairwise rows:", len(pairwise_df))
    display(pairwise_df.head(20))
else:
    pairwise_df = pd.DataFrame()
    print("Pairwise analysis skipped. Set RUN_PAIRWISE_ANALYSIS=True to enable.")

In [ ]:
def build_open3d_consistency_geometries(
    point_df: pd.DataFrame,
    points3D_dict: Dict[int, object],
    max_vector_count: int = 800,
):
    if not OPEN3D_AVAILABLE:
        return []

    required = ["point3D_id", "median_angle_deg", "consensus_nx", "consensus_ny", "consensus_nz"]
    for c in required:
        if c not in point_df.columns:
            raise KeyError(f"Missing column: {c}")

    subset = point_df[np.isfinite(point_df["median_angle_deg"])].copy()
    if subset.empty:
        return []

    xyzs = []
    errs = []
    cons = []
    for row in subset.itertuples(index=False):
        pt = points3D_dict.get(int(row.point3D_id))
        if pt is None:
            continue
        xyz = np.asarray(pt.xyz, dtype=np.float64)
        n = np.asarray([row.consensus_nx, row.consensus_ny, row.consensus_nz], dtype=np.float64)
        n = safe_normalize(n)
        if n is None:
            continue
        xyzs.append(xyz)
        errs.append(float(row.median_angle_deg))
        cons.append(n)

    if not xyzs:
        return []

    xyzs = np.asarray(xyzs, dtype=np.float64)
    errs = np.asarray(errs, dtype=np.float64)
    cons = np.asarray(cons, dtype=np.float64)

    scale = float(np.percentile(errs, 95)) if len(errs) else 1.0
    if not np.isfinite(scale) or scale < 1e-6:
        scale = float(max(np.max(errs), 1.0))
    t = np.clip(errs / scale, 0.0, 1.0)

    colors = np.column_stack([t, 1.0 - t, np.zeros_like(t)])

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyzs)
    pcd.colors = o3d.utility.Vector3dVector(colors)

    geoms = [pcd]

    if max_vector_count > 0 and len(xyzs) > 0:
        step = max(1, len(xyzs) // int(max_vector_count))
        starts = xyzs[::step]
        dirs = cons[::step]

        bbox_diag = float(np.linalg.norm(xyzs.max(axis=0) - xyzs.min(axis=0)))
        vec_len = 0.02 * bbox_diag if bbox_diag > 0 else 0.05

        line_points = []
        lines = []
        line_colors = []
        for i, (s, d) in enumerate(zip(starts, dirs)):
            p0 = s
            p1 = s + vec_len * d
            line_points.append(p0)
            line_points.append(p1)
            lines.append([2 * i, 2 * i + 1])
            line_colors.append([0.1, 0.1, 1.0])

        line_set = o3d.geometry.LineSet()
        line_set.points = o3d.utility.Vector3dVector(np.asarray(line_points))
        line_set.lines = o3d.utility.Vector2iVector(np.asarray(lines, dtype=np.int32))
        line_set.colors = o3d.utility.Vector3dVector(np.asarray(line_colors, dtype=np.float64))
        geoms.append(line_set)

    return geoms


def show_open3d_geometries(geometries, backend: str = "auto", window_name: str = "Open3D"):
    if not OPEN3D_AVAILABLE:
        print("Open3D is not installed; skipping 3D visualization.")
        return None
    if not geometries:
        print("No valid geometries to visualize.")
        return None

    backend = str(backend).strip().lower()
    if backend not in {"auto", "notebook", "window"}:
        raise ValueError(f"Unsupported OPEN3D_VIS_BACKEND='{backend}'. Use auto/notebook/window.")

    try_notebook = backend in {"auto", "notebook"}
    if try_notebook and IN_NOTEBOOK and OPEN3D_WEB_DRAW_AVAILABLE:
        try:
            try:
                o3d.visualization.webrtc_server.enable_webrtc()
            except Exception:
                pass
            try:
                return o3d_web_draw(geometries, name=window_name, show_ui=True)
            except TypeError:
                return o3d_web_draw(geometries)
        except Exception as exc:
            if backend == "notebook":
                print("Open3D notebook visualization failed:", exc)
                return None
            print("Open3D notebook visualization failed; falling back to desktop window:", exc)

    if backend == "notebook":
        print("Notebook visualization requested but unavailable. Falling back is disabled for backend='notebook'.")
        print("Requirements: Jupyter kernel + open3d.web_visualizer support.")
        return None

    try:
        o3d.visualization.draw_geometries(
            geometries,
            window_name=window_name,
        )
    except Exception as exc:
        print("Open3D window visualization failed:", exc)

    return None


if OPEN3D_AVAILABLE:
    geoms = build_open3d_consistency_geometries(
        point_df=point_df,
        points3D_dict=points3D_by_id,
        max_vector_count=OPEN3D_MAX_VECTOR_COUNT,
    )
    print("Prepared Open3D geometries:", len(geoms))
    if ENABLE_OPEN3D_VIS:
        vis_widget = show_open3d_geometries(
            geoms,
            backend=OPEN3D_VIS_BACKEND,
            window_name="Normal Consistency (green=consistent, red=inconsistent)",
        )
        vis_widget
    else:
        print("Open3D visualization disabled; set ENABLE_OPEN3D_VIS=True to enable.")
else:
    print("Open3D is not installed; skipping 3D visualization cell.")


In [ ]:
OUTPUT_DIR = Path("outputs/normal_consistency")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

obs_csv = OUTPUT_DIR / "observations.csv"
point_csv = OUTPUT_DIR / "points.csv"
image_csv = OUTPUT_DIR / "images.csv"
match_csv = OUTPUT_DIR / "image_normal_matches.csv"

obs_df.to_csv(obs_csv, index=False)
point_df.to_csv(point_csv, index=False)
image_df.to_csv(image_csv, index=False)
match_df.to_csv(match_csv, index=False)

print("Saved:")
print(" ", obs_csv)
print(" ", point_csv)
print(" ", image_csv)
print(" ", match_csv)